In [1]:
try:
    import pgmpy
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, Conv2D, Flatten, LSTM, Input, Bidirectional
    from tensorflow.keras.optimizers import Adam
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pgmpy", "matplotlib", "seaborn", "tensorflow", "-q"])
    import pgmpy
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, Conv2D, Flatten, LSTM, Input, Bidirectional
    from tensorflow.keras.optimizers import Adam

import urllib.request
import pandas as pd
from google.colab import files
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time
import traceback
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
import networkx as nx
import itertools
import json

In [2]:
def load_dataset(method="upload", file_path=None, url=None):
    try:
        if method == "upload":
            print("Please upload your dataset file (CSV or Excel)")
            uploaded = files.upload()
            for filename, content in uploaded.items():
                with open(filename, 'wb') as f:
                    f.write(content)
                if filename.endswith('.csv'):
                    df = pd.read_csv(filename)
                elif filename.endswith('.xlsx'):
                    df = pd.read_excel(filename)
                else:
                    raise ValueError("Unsupported file format. Use CSV or Excel.")
                print(f"Loaded dataset {filename} with shape {df.shape}")
                return df
        elif method == "drive":
            from google.colab import drive
            drive.mount('/content/drive')
            if file_path and os.path.exists(file_path):
                df = pd.read_csv(file_path)
                print(f"Loaded dataset from Google Drive {file_path} with shape {df.shape}")
                return df
            else:
                raise ValueError("File path not found in Google Drive")
        elif method == "url":
            if url:
                output_file = "downloaded_dataset.csv"
                try:
                    urllib.request.urlretrieve(url, output_file)
                    df = pd.read_csv(output_file, low_memory=False)
                    print(f"Loaded dataset from URL {url} with shape {df.shape}")
                    return df
                except Exception as e:
                    print(f"Error downloading from URL {url}: {e}")
                    return None
            else:
                raise ValueError("URL not provided")
        else:
            raise ValueError("Invalid method. Use 'upload', 'drive', or 'url'")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        traceback.print_exc()
        return None

In [3]:
def preprocess_dataset(df, missing_strategy='mean', output_path="preprocessed_dataset.csv"):
    try:
        df_clean = df.copy()
        pd.set_option('future.no_silent_downcasting', True)

        # Clean Total Costs and Total Charges
        if 'Total Costs' in df_clean.columns:
            df_clean['Total Costs'] = df_clean['Total Costs'].astype(str).str.replace(r'[\$,]', '', regex=True)
            df_clean['Total Costs'] = pd.to_numeric(df_clean['Total Costs'], errors='coerce')
        if 'Total Charges' in df_clean.columns:
            df_clean['Total Charges'] = df_clean['Total Charges'].astype(str).str.replace(r'[\$,]', '', regex=True)
            df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce')
        print("Cleaned Total Costs and Total Charges columns")

        # Create binary 'Sick' label
        if 'APR Severity of Illness Code' in df_clean.columns:
            df_clean['APR Severity of Illness Code'] = pd.to_numeric(df_clean['APR Severity of Illness Code'], errors='coerce')
            df_clean['Sick'] = (df_clean['APR Severity of Illness Code'] >= 3).astype(int)
            print("Created binary 'Sick' column based on APR Severity of Illness Code >= 3")

        # Create 'Survive' column
        if 'APR Risk of Mortality' in df_clean.columns:
            severity_map = {'Minor': 1, 'Moderate': 2, 'Major': 3, 'Extreme': 4}
            df_clean['APR Risk of Mortality'] = df_clean['APR Risk of Mortality'].replace(severity_map)
            df_clean['APR Risk of Mortality'] = pd.to_numeric(df_clean['APR Risk of Mortality'], errors='coerce')
            nan_count = df_clean['APR Risk of Mortality'].isna().sum()
            if nan_count > 0:
                median_risk = df_clean['APR Risk of Mortality'].median()
                df_clean['APR Risk of Mortality'] = df_clean['APR Risk of Mortality'].fillna(median_risk if not pd.isna(median_risk) else 1)
                print(f"Filled {nan_count} NaN values in 'APR Risk of Mortality' with median: {median_risk}")
            df_clean['Survive'] = (df_clean['APR Risk of Mortality'] < 3).astype(int)
            print(f"Created binary 'Survive' column")

        # Create other binary outcomes
        if 'Type of Admission' in df_clean.columns:
            df_clean['Need Hospitalization'] = (df_clean['Type of Admission'].astype(str).str.contains('Emergency', na=False, case=False)).astype(int)
            print("Created binary 'Need Hospitalization' column")
        if 'APR Severity of Illness Code' in df_clean.columns:
            df_clean['Need Medication'] = (df_clean['APR Severity of Illness Code'] > 1).astype(int)
            print("Created binary 'Need Medication' column")
        if 'Principal Procedure Code' in df_clean.columns:
            df_clean['Need Surgery'] = df_clean['Principal Procedure Code'].notna().astype(int)
            print("Created binary 'Need Surgery' column")

        # Discretize Total Costs
        if 'Total Costs' in df_clean.columns:
            valid_costs = df_clean['Total Costs'].dropna()
            if len(valid_costs) > 0:
                df_clean['Cost Range'] = pd.cut(df_clean['Total Costs'], bins=4, labels=["Low", "Medium", "High", "Very High"], duplicates='drop')
                print("Discretized 'Total Costs' into 'Cost Range'")

        # Discretize Length of Stay
        if 'Length of Stay' in df_clean.columns:
            df_clean['Length of Stay'] = pd.to_numeric(df_clean['Length of Stay'], errors='coerce')
            valid_los = df_clean['Length of Stay'].dropna()
            if len(valid_los) > 0:
                df_clean['Stay Range'] = pd.cut(df_clean['Length of Stay'], bins=4, labels=["Short", "Medium", "Long", "Very Long"], duplicates='drop')
                print("Discretized 'Length of Stay' into 'Stay Range'")

        # Create Age Group
        if 'Age Group' in df_clean.columns:
            df_clean['Age Group'] = df_clean['Age Group'].astype(str)
            print("Kept 'Age Group' as categorical variable")
        elif 'Age' in df_clean.columns:
            df_clean['Age'] = pd.to_numeric(df_clean['Age'], errors='coerce')
            df_clean['Age Group'] = pd.cut(df_clean['Age'], bins=[0, 18, 35, 50, 65, 100], labels=["0-18", "19-35", "36-50", "51-65", "66+"], duplicates='drop')
            print("Created 'Age Group' from 'Age' column")

        # Handle missing values
        for column in df_clean.columns:
            null_count = df_clean[column].isnull().sum()
            if null_count > 0:
                if df_clean[column].dtype in [np.float64, np.int64]:
                    if missing_strategy == 'mean':
                        df_clean[column] = df_clean[column].fillna(df_clean[column].mean())
                    elif missing_strategy == 'median':
                        df_clean[column] = df_clean[column].fillna(df_clean[column].median())
                    else:
                        df_clean[column] = df_clean[column].fillna(0)
                else:
                    mode_val = df_clean[column].mode()
                    df_clean[column] = df_clean[column].fillna(mode_val[0] if len(mode_val) > 0 else 'Unknown')
        print(f"Missing values handled. Total null count: {df_clean.isnull().sum().sum()}")

        # Encode categorical variables
        le = LabelEncoder()
        categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns
        for column in categorical_cols:
            df_clean[column] = le.fit_transform(df_clean[column].astype(str))
        print(f"Encoded {len(categorical_cols)} categorical variables")

        # Scale numerical features
        scaler = StandardScaler()
        numerical_cols = df_clean.select_dtypes(include=[np.float64, np.int64]).columns
        exclude_cols = ['Sick', 'Survive', 'Need Hospitalization', 'Need Medication', 'Need Surgery']
        numerical_cols = [col for col in numerical_cols if col not in exclude_cols]
        if len(numerical_cols) > 0:
            df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
            print(f"Scaled {len(numerical_cols)} numerical features")

        # Sort for Markov analysis
        if 'APR Severity of Illness Code' in df_clean.columns:
            df_clean = df_clean.sort_values(by='APR Severity of Illness Code')
            print("Sorted dataset by 'APR Severity of Illness Code'")

        # Save preprocessed dataset
        df_clean.to_csv(output_path, index=False)
        print(f"Preprocessed dataset saved as {output_path}")
        return df_clean
    except Exception as e:
        print(f"Error preprocessing dataset: {e}")
        traceback.print_exc()
        return None

In [4]:
def markov_chain_analysis(df, state_column, outcome_columns=None):
    try:
        if state_column not in df.columns:
            print(f"Error: {state_column} not found in dataset")
            return None

        states = df[state_column].unique()
        n_states = len(states)
        state_map = {state: idx for idx, state in enumerate(states)}

        transition_matrix = np.zeros((n_states, n_states))
        df_sorted = df.sort_values(by=['APR Severity of Illness Code'])
        for i in range(len(df_sorted) - 1):
            current_state = df_sorted[state_column].iloc[i]
            next_state = df_sorted[state_column].iloc[i + 1]
            if pd.notna(current_state) and pd.notna(next_state):
                transition_matrix[state_map[current_state], state_map[next_state]] += 1

        row_sums = transition_matrix.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        transition_matrix = transition_matrix / row_sums

        print(f"Transition Matrix for {state_column}:")
        print(pd.DataFrame(transition_matrix, index=states, columns=states))

        outcome_results = {}
        if outcome_columns:
            for outcome in outcome_columns:
                if outcome in df.columns:
                    transition_outcome = pd.crosstab(df[state_column], df[outcome], normalize='index')
                    outcome_results[outcome] = transition_outcome
                    print(f"Transition probabilities for {outcome} by {state_column}:")
                    print(transition_outcome)

        return {'transition_matrix': transition_matrix, 'states': states, 'outcome_results': outcome_results}
    except Exception as e:
        print(f"Error in Markov chain analysis: {e}")
        traceback.print_exc()
        return None

In [5]:
def neural_network_with_backpropagation(df, target_column, epochs=50, batch_size=32):
    try:
        sample_size = 1000
        df_sample = df.sample(n=sample_size, random_state=42)
        X = df_sample.drop(columns=[target_column]).values
        y = df_sample[target_column].values

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        model = Sequential([
            Input(shape=(X_train.shape[1],)),
            Dense(128, activation='relu'),
            Dropout(0.3),
            Dense(64, activation='relu'),
            Dropout(0.3),
            Dense(32, activation='relu'),
            Dense(1, activation='sigmoid')
        ])

        model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
        history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)

        y_pred_prob = model.predict(X_test, verbose=0)
        y_pred = (y_pred_prob > 0.5).astype(int).flatten()

        metrics = evaluate_metrics(y_test, y_pred, model_name="Neural Network", generate_chart=True)
        print(f"Neural Network with Backpropagation Metrics:")
        print(f"Test Loss: {model.evaluate(X_test, y_test, verbose=0)[0]:.4f}")

        plt.figure(figsize=(10, 6))
        plt.plot(history.history['accuracy'], label='Training Accuracy')
        plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
        plt.title('Neural Network Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.savefig('nn_accuracy.png')
        plt.close()

        return model, history, y_test, y_pred, metrics
    except Exception as e:
        print(f"Error in neural network training: {e}")
        traceback.print_exc()
        return None, None, None, None, None

In [6]:
def enhanced_cnn_for_tabular(df, target_column):
    try:
        sample_size = 1000
        df_sample = df.sample(n=sample_size, random_state=42)
        X = df_sample.drop(columns=[target_column]).values
        y = df_sample[target_column].values

        n_features = X.shape[1]
        side = int(np.ceil(np.sqrt(n_features)))
        pad_size = side * side - n_features
        X_padded = np.pad(X, ((0, 0), (0, pad_size)), mode='constant')
        X_reshaped = X_padded.reshape(-1, side, side, 1)

        X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y, test_size=0.2, random_state=42)

        model = Sequential([
            Input(shape=(side, side, 1)),
            Conv2D(64, (3, 3), activation='relu', padding='same'),
            Conv2D(32, (3, 3), activation='relu', padding='same'),
            Conv2D(16, (3, 3), activation='relu', padding='same'),
            Flatten(),
            Dense(128, activation='relu'),
            Dropout(0.4),
            Dense(64, activation='relu'),
            Dropout(0.4),
            Dense(1, activation='sigmoid')
        ])

        model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
        history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)

        y_pred_prob = model.predict(X_test, verbose=0)
        y_pred = (y_pred_prob > 0.5).astype(int).flatten()

        metrics = evaluate_metrics(y_test, y_pred, model_name="Enhanced CNN", generate_chart=True)
        print(f"Enhanced CNN for Tabular Data Metrics:")
        print(f"Test Loss: {model.evaluate(X_test, y_test, verbose=0)[0]:.4f}")

        plt.figure(figsize=(10, 6))
        plt.plot(history.history['accuracy'], label='Training Accuracy')
        plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
        plt.title('Enhanced CNN Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.savefig('enhanced_cnn_accuracy.png')
        plt.close()

        return model, history, y_test, y_pred, metrics
    except Exception as e:
        print(f"Error in Enhanced CNN for tabular data: {e}")
        traceback.print_exc()
        return None, None, None, None, None

In [7]:
def enhanced_rnn_for_sequential(df, target_column, sequence_length=3):
    try:
        sample_size = 1000
        df_sample = df.sample(n=sample_size, random_state=42)
        X = df_sample.drop(columns=[target_column]).values
        y = df_sample[target_column].values

        X_seq = []
        y_seq = []
        for i in range(len(X) - sequence_length):
            X_seq.append(X[i:i + sequence_length])
            y_seq.append(y[i + sequence_length])
        X_seq = np.array(X_seq)
        y_seq = np.array(y_seq)

        X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

        model = Sequential([
            Input(shape=(sequence_length, X.shape[1])),
            Bidirectional(LSTM(64, return_sequences=True)),
            Dropout(0.4),
            Bidirectional(LSTM(32)),
            Dropout(0.4),
            Dense(32, activation='relu'),
            Dense(1, activation='sigmoid')
        ])

        model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
        history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)

        y_pred_prob = model.predict(X_test, verbose=0)
        y_pred = (y_pred_prob > 0.5).astype(int).flatten()

        metrics = evaluate_metrics(y_test, y_pred, model_name="Enhanced RNN (Bidirectional LSTM)", generate_chart=True)
        print(f"Enhanced RNN for Sequential Analysis Metrics:")
        print(f"Test Loss: {model.evaluate(X_test, y_test, verbose=0)[0]:.4f}")

        plt.figure(figsize=(10, 6))
        plt.plot(history.history['accuracy'], label='Training Accuracy')
        plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
        plt.title('Enhanced RNN Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.savefig('enhanced_rnn_accuracy.png')
        plt.close()

        return model, history, y_test, y_pred, metrics
    except Exception as e:
        print(f"Error in Enhanced RNN for sequential analysis: {e}")
        traceback.print_exc()
        return None, None, None, None, None

In [8]:
def custom_hyperparameter_tuning(df, target_column):
    try:
        sample_size = 1000
        df_sample = df.sample(n=sample_size, random_state=42)
        X = df_sample.drop(columns=[target_column]).values
        y = df_sample[target_column].values

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        param_grid = {
            'hidden_size': [64, 128],
            'dropout_rate': [0.2, 0.3],
            'learning_rate': [0.001, 0.01],
            'batch_size': [32, 64],
            'epochs': [20, 50]
        }

        best_accuracy = 0
        best_params = None
        best_model = None
        best_y_pred = None

        for params in itertools.product(*param_grid.values()):
            param_dict = dict(zip(param_grid.keys(), params))
            model = Sequential([
                Input(shape=(X_train.shape[1],)),
                Dense(param_dict['hidden_size'], activation='relu'),
                Dropout(param_dict['dropout_rate']),
                Dense(param_dict['hidden_size'] // 2, activation='relu'),
                Dropout(param_dict['dropout_rate']),
                Dense(1, activation='sigmoid')
            ])
            model.compile(optimizer=Adam(learning_rate=param_dict['learning_rate']), loss='binary_crossentropy', metrics=['accuracy'])
            model.fit(X_train, y_train, epochs=param_dict['epochs'], batch_size=param_dict['batch_size'], validation_split=0.2, verbose=0)
            _, accuracy = model.evaluate(X_test, y_test, verbose=0)
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_params = param_dict
                best_model = model
                y_pred_prob = model.predict(X_test, verbose=0)
                best_y_pred = (y_pred_prob > 0.5).astype(int).flatten()

        metrics = evaluate_metrics(y_test, best_y_pred, model_name="Tuned Neural Network", generate_chart=True)
        print(f"Custom Hyperparameter Tuning Results:")
        print(f"Best Test Accuracy: {best_accuracy:.4f}")
        print(f"Best Parameters: {best_params}")

        return best_model, best_params, y_test, best_y_pred, metrics
    except Exception as e:
        print(f"Error in custom hyperparameter tuning: {e}")
        traceback.print_exc()
        return None, None, None, None, None

In [9]:
def cnn_for_tabular(df, target_column):
    try:
        sample_size = 1000
        df_sample = df.sample(n=sample_size, random_state=42)
        X = df_sample.drop(columns=[target_column]).values
        y = df_sample[target_column].values

        n_features = X.shape[1]
        side = int(np.ceil(np.sqrt(n_features)))
        pad_size = side * side - n_features
        X_padded = np.pad(X, ((0, 0), (0, pad_size)), mode='constant')
        X_reshaped = X_padded.reshape(-1, side, side, 1)

        X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y, test_size=0.2, random_state=42)

        model = Sequential([
            Input(shape=(side, side, 1)),
            Conv2D(32, (3, 3), activation='relu', padding='same'),
            Conv2D(16, (3, 3), activation='relu', padding='same'),
            Flatten(),
            Dense(64, activation='relu'),
            Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])

        model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
        history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=0)

        y_pred_prob = model.predict(X_test, verbose=0)
        y_pred = (y_pred_prob > 0.5).astype(int).flatten()

        metrics = evaluate_metrics(y_test, y_pred, model_name="CNN", generate_chart=True)
        print(f"CNN for Tabular Data Metrics:")
        print(f"Test Loss: {model.evaluate(X_test, y_test, verbose=0)[0]:.4f}")

        plt.figure(figsize=(10, 6))
        plt.plot(history.history['accuracy'], label='Training Accuracy')
        plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
        plt.title('CNN Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.savefig('cnn_accuracy.png')
        plt.close()

        return model, history, y_test, y_pred, metrics
    except Exception as e:
        print(f"Error in CNN for tabular data: {e}")
        traceback.print_exc()
        return None, None, None, None, None

In [10]:
def rnn_for_sequential(df, target_column, sequence_length=3):
    try:
        sample_size = 1000
        df_sample = df.sample(n=sample_size, random_state=42)
        X = df_sample.drop(columns=[target_column]).values
        y = df_sample[target_column].values

        X_seq = []
        y_seq = []
        for i in range(len(X) - sequence_length):
            X_seq.append(X[i:i + sequence_length])
            y_seq.append(y[i + sequence_length])
        X_seq = np.array(X_seq)
        y_seq = np.array(y_seq)

        X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

        model = Sequential([
            Input(shape=(sequence_length, X.shape[1])),
            LSTM(64, return_sequences=False),
            Dropout(0.3),
            Dense(32, activation='relu'),
            Dense(1, activation='sigmoid')
        ])

        model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
        history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=0)

        y_pred_prob = model.predict(X_test, verbose=0)
        y_pred = (y_pred_prob > 0.5).astype(int).flatten()

        metrics = evaluate_metrics(y_test, y_pred, model_name="RNN", generate_chart=True)
        print(f"RNN for Sequential Analysis Metrics:")
        print(f"Test Loss: {model.evaluate(X_test, y_test, verbose=0)[0]:.4f}")

        plt.figure(figsize=(10, 6))
        plt.plot(history.history['accuracy'], label='Training Accuracy')
        plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
        plt.title('RNN Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.savefig('rnn_accuracy.png')
        plt.close()

        return model, history, y_test, y_pred, metrics
    except Exception as e:
        print(f"Error in RNN for sequential analysis: {e}")
        traceback.print_exc()
        return None, None, None, None, None

In [11]:
def svm_classification(df, target_column, file_path=None):
    try:
        sample_size = 1000
        df_sample = df.sample(n=sample_size, random_state=42)
        X = df_sample.drop(columns=[target_column]).values
        y = df_sample[target_column].values

        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X)

        svm_model = SVC(kernel='linear', random_state=42)
        svm_model.fit(X_pca, y)

        X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)
        y_pred = svm_model.predict(X_test)

        metrics = evaluate_metrics(y_test, y_pred, model_name="SVM", generate_chart=True)
        print(f"SVM Classification Metrics:")
        print(f"Training accuracy: {svm_model.score(X_train, y_train):.4f}")

        return svm_model, y_test, y_pred, metrics
    except Exception as e:
        print(f"Error in SVM classification: {e}")
        traceback.print_exc()
        return None, None, None, None

def visualize_svm(svm_model, df, target_column):
    try:
        sample_size = 1000
        df_sample = df.sample(n=sample_size, random_state=42)
        X = df_sample.drop(columns=[target_column]).values
        y = df_sample[target_column].values

        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X)

        h = 0.02
        x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
        y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

        Z = svm_model.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)

        plt.figure(figsize=(10, 8))
        plt.contourf(xx, yy, Z, cmap=plt.cm.RdYlBu, alpha=0.4)
        plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap=plt.cm.RdYlBu, edgecolor='k')
        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
        plt.title('SVM Decision Boundary (Sick vs. Not Sick)')
        plt.savefig('svm_decision_boundary.png')
        plt.close()
        print("SVM decision boundary visualization saved as 'svm_decision_boundary.png'.")
    except Exception as e:
        print(f"Error in SVM visualization: {e}")
        traceback.print_exc()

In [12]:
def evaluate_metrics(y_test, y_pred, model_name="Model", file_path=None, generate_chart=False):
    try:
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        print(f"{model_name} Evaluation Metrics:")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")

        if generate_chart:
            chart_data = {
                "type": "bar",
                "data": {
                    "labels": ["Accuracy", "Precision", "Recall", "F1-Score"],
                    "datasets": [{
                        "label": f"{model_name} Metrics",
                        "data": [accuracy, precision, recall, f1],
                        "backgroundColor": ["#FF6384", "#36A2EB", "#FFCE56", "#4BC0C0"],
                        "borderColor": ["#FF6384", "#36A2EB", "#FFCE56", "#4BC0C0"],
                        "borderWidth": 1
                    }]
                },
                "options": {
                    "scales": {
                        "y": {
                            "beginAtZero": True,
                            "title": {"display": True, "text": "Score"}
                        }
                    },
                    "plugins": {
                        "title": {"display": True, "text": f"{model_name} Evaluation Metrics"}
                    }
                }
            }
            print("Chart for evaluation metrics:")
            print(f"```chartjs\n{json.dumps(chart_data)}\n```")

        return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}
    except Exception as e:
        print(f"Error in evaluation metrics for {model_name}: {e}")
        traceback.print_exc()
        return None

In [13]:
def bayesian_network_analysis(df, bayesian_columns, bayesian_edges, sample_size=1000):
    try:
        df_sample = df[bayesian_columns].sample(n=sample_size, random_state=42)
        df_sample = df_sample.dropna()

        model = DiscreteBayesianNetwork(bayesian_edges)
        model.fit(df_sample, estimator=MaximumLikelihoodEstimator)

        inference = VariableElimination(model)
        print("Bayesian Network Analysis:")
        for node in model.nodes():
            print(f"CPD for {node}:")
            print(model.get_cpds(node))

        if 'Sick' in bayesian_columns:
            evidence = {col: df_sample[col].mode()[0] for col in bayesian_columns if col != 'Sick'}
            prob_sick = inference.query(variables=['Sick'], evidence=evidence)
            print(f"Inference: Probability of Sick given {evidence}:")
            print(prob_sick)

        G = nx.DiGraph()
        G.add_edges_from(bayesian_edges)
        pos = nx.spring_layout(G)
        plt.figure(figsize=(8, 6))
        nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000, font_size=10, font_weight='bold')
        plt.title('Bayesian Network Structure')
        plt.savefig('bayesian_network.png')
        plt.close()
        print("Bayesian network visualization saved as 'bayesian_network.png'.")

        return model, inference
    except Exception as e:
        print(f"Error in Bayesian network analysis: {e}")
        traceback.print_exc()
        return None, None

In [14]:
def monte_carlo_cost(df, num_simulations=1000, sample_size=1000):
    try:
        if 'Total Costs' not in df.columns:
            print("Error: 'Total Costs' column not found in dataset")
            return None

        df_valid = df.dropna(subset=['Total Costs'])
        if len(df_valid) == 0:
            print("Error: No valid data in 'Total Costs' column")
            return None

        mean_costs = []
        for _ in range(num_simulations):
            sample = df_valid['Total Costs'].sample(n=sample_size, replace=True)
            mean = sample.mean()
            mean_costs.append(mean)

        mean_value = np.mean(mean_costs)
        std_value = np.std(mean_costs)
        ci_low = np.percentile(mean_costs, 2.5)
        ci_high = np.percentile(mean_costs, 97.5)

        print(f"Monte Carlo Simulation for Total Costs:")
        print(f"Estimated Mean Cost: ${mean_value:.2f}")
        print(f"Standard Deviation: ${std_value:.2f}")
        print(f"95% Confidence Interval: [${ci_low:.2f}, ${ci_high:.2f}]")

        plt.figure(figsize=(8, 6))
        sns.histplot(mean_costs, kde=True, color='blue')
        plt.title('Monte Carlo Simulation: Distribution of Mean Costs')
        plt.xlabel('Mean Cost ($)')
        plt.ylabel('Frequency')
        plt.savefig('monte_carlo_costs.png')
        plt.close()
        print("Monte Carlo cost distribution saved as 'monte_carlo_costs.png'.")

        return {
            'mean_value': mean_value,
            'std_value': std_value,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'simulations': mean_costs
        }
    except Exception as e:
        print(f"Error in Monte Carlo simulation for cost: {e}")
        traceback.print_exc()
        return None

def monte_carlo_survival(df, num_simulations=1000, sample_size=1000):
    try:
        if 'Survive' not in df.columns:
            print("Error: 'Survive' column not found in dataset")
            return None

        df_valid = df.dropna(subset=['Survive'])
        if len(df_valid) == 0:
            print("Error: No valid data in 'Survive' column")
            return None

        survival_probs = []
        for _ in range(num_simulations):
            sample = df_valid['Survive'].sample(n=sample_size, replace=True)
            prob = sample.mean()
            survival_probs.append(prob)

        mean_prob = np.mean(survival_probs)
        std_prob = np.std(survival_probs)
        ci_low = np.percentile(survival_probs, 2.5)
        ci_high = np.percentile(survival_probs, 97.5)

        print(f"Monte Carlo Simulation for Survival:")
        print(f"Estimated Survival Probability: {mean_prob:.4f}")
        print(f"Standard Deviation: {std_prob:.4f}")
        print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")

        plt.figure(figsize=(8, 6))
        sns.histplot(survival_probs, kde=True, color='green')
        plt.title('Monte Carlo Simulation: Distribution of Survival Probabilities')
        plt.xlabel('Survival Probability')
        plt.ylabel('Frequency')
        plt.savefig('monte_carlo_survival.png')
        plt.close()
        print("Monte Carlo survival distribution saved as 'monte_carlo_survival.png'.")

        return {
            'mean_prob': mean_prob,
            'std_prob': std_prob,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'simulations': survival_probs
        }
    except Exception as e:
        print(f"Error in Monte Carlo simulation for survival: {e}")
        traceback.print_exc()
        return None

def monte_carlo_hospitalization(df, num_simulations=1000, sample_size=1000):
    try:
        if 'Need Hospitalization' not in df.columns:
            print("Error: 'Need Hospitalization' column not found in dataset")
            return None

        df_valid = df.dropna(subset=['Need Hospitalization'])
        if len(df_valid) == 0:
            print("Error: No valid data in 'Need Hospitalization' column")
            return None

        probs = []
        for _ in range(num_simulations):
            sample = df_valid['Need Hospitalization'].sample(n=sample_size, replace=True)
            prob = sample.mean()
            probs.append(prob)

        mean_prob = np.mean(probs)
        std_prob = np.std(probs)
        ci_low = np.percentile(probs, 2.5)
        ci_high = np.percentile(probs, 97.5)

        print(f"Monte Carlo Simulation for Need Hospitalization:")
        print(f"Estimated Probability: {mean_prob:.4f}")
        print(f"Standard Deviation: {std_prob:.4f}")
        print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")

        plt.figure(figsize=(8, 6))
        sns.histplot(probs, kde=True, color='orange')
        plt.title('Monte Carlo Simulation: Distribution of Hospitalization Probabilities')
        plt.xlabel('Hospitalization Probability')
        plt.ylabel('Frequency')
        plt.savefig('monte_carlo_hospitalization.png')
        plt.close()
        print("Monte Carlo hospitalization distribution saved as 'monte_carlo_hospitalization.png'.")

        return {
            'mean_prob': mean_prob,
            'std_prob': std_prob,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'simulations': probs
        }
    except Exception as e:
        print(f"Error in Monte Carlo simulation for hospitalization: {e}")
        traceback.print_exc()
        return None

def monte_carlo_medication(df, num_simulations=1000, sample_size=1000):
    try:
        if 'Need Medication' not in df.columns:
            print("Error: 'Need Medication' column not found in dataset")
            return None

        df_valid = df.dropna(subset=['Need Medication'])
        if len(df_valid) == 0:
            print("Error: No valid data in 'Need Medication' column")
            return None

        probs = []
        for _ in range(num_simulations):
            sample = df_valid['Need Medication'].sample(n=sample_size, replace=True)
            prob = sample.mean()
            probs.append(prob)

        mean_prob = np.mean(probs)
        std_prob = np.std(probs)
        ci_low = np.percentile(probs, 2.5)
        ci_high = np.percentile(probs, 97.5)

        print(f"Monte Carlo Simulation for Need Medication:")
        print(f"Estimated Probability: {mean_prob:.4f}")
        print(f"Standard Deviation: {std_prob:.4f}")
        print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")

        plt.figure(figsize=(8, 6))
        sns.histplot(probs, kde=True, color='purple')
        plt.title('Monte Carlo Simulation: Distribution of Medication Probabilities')
        plt.xlabel('Medication Probability')
        plt.ylabel('Frequency')
        plt.savefig('monte_carlo_medication.png')
        plt.close()
        print("Monte Carlo medication distribution saved as 'monte_carlo_medication.png'.")

        return {
            'mean_prob': mean_prob,
            'std_prob': std_prob,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'simulations': probs
        }
    except Exception as e:
        print(f"Error in Monte Carlo simulation for medication: {e}")
        traceback.print_exc()
        return None

def monte_carlo_length_of_stay(df, num_simulations=1000, sample_size=1000):
    try:
        if 'Length of Stay' not in df.columns:
            print("Error: 'Length of Stay' column not found in dataset")
            return None

        df_valid = df.dropna(subset=['Length of Stay'])
        if len(df_valid) == 0:
            print("Error: No valid data in 'Length of Stay' column")
            return None

        mean_los = []
        for _ in range(num_simulations):
            sample = df_valid['Length of Stay'].sample(n=sample_size, replace=True)
            mean = sample.mean()
            mean_los.append(mean)

        mean_value = np.mean(mean_los)
        std_value = np.std(mean_los)
        ci_low = np.percentile(mean_los, 2.5)
        ci_high = np.percentile(mean_los, 97.5)

        print(f"Monte Carlo Simulation for Length of Stay:")
        print(f"Estimated Mean Length of Stay: {mean_value:.2f} days")
        print(f"Standard Deviation: {std_value:.2f} days")
        print(f"95% Confidence Interval: [{ci_low:.2f}, {ci_high:.2f}] days")

        plt.figure(figsize=(8, 6))
        sns.histplot(mean_los, kde=True, color='red')
        plt.title('Monte Carlo Simulation: Distribution of Mean Length of Stay')
        plt.xlabel('Mean Length of Stay (days)')
        plt.ylabel('Frequency')
        plt.savefig('monte_carlo_length_of_stay.png')
        plt.close()
        print("Monte Carlo length of stay distribution saved as 'monte_carlo_length_of_stay.png'.")

        return {
            'mean_value': mean_value,
            'std_value': std_value,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'simulations': mean_los
        }
    except Exception as e:
        print(f"Error in Monte Carlo simulation for length of stay: {e}")
        traceback.print_exc()
        return None

def monte_carlo_surgery(df, num_simulations=1000, sample_size=1000):
    try:
        if 'Need Surgery' not in df.columns:
            print("Error: 'Need Surgery' column not found in dataset")
            return None

        df_valid = df.dropna(subset=['Need Surgery'])
        if len(df_valid) == 0:
            print("Error: No valid data in 'Need Surgery' column")
            return None

        probs = []
        for _ in range(num_simulations):
            sample = df_valid['Need Surgery'].sample(n=sample_size, replace=True)
            prob = sample.mean()
            probs.append(prob)

        mean_prob = np.mean(probs)
        std_prob = np.std(probs)
        ci_low = np.percentile(probs, 2.5)
        ci_high = np.percentile(probs, 97.5)

        print(f"Monte Carlo Simulation for Need Surgery:")
        print(f"Estimated Probability: {mean_prob:.4f}")
        print(f"Standard Deviation: {std_prob:.4f}")
        print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")

        plt.figure(figsize=(8, 6))
        sns.histplot(probs, kde=True, color='teal')
        plt.title('Monte Carlo Simulation: Distribution of Surgery Probabilities')
        plt.xlabel('Surgery Probability')
        plt.ylabel('Frequency')
        plt.savefig('monte_carlo_surgery.png')
        plt.close()
        print("Monte Carlo surgery distribution saved as 'monte_carlo_surgery.png'.")

        return {
            'mean_prob': mean_prob,
            'std_prob': std_prob,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'simulations': probs
        }
    except Exception as e:
        print(f"Error in Monte Carlo simulation for surgery: {e}")
        traceback.print_exc()
        return None

In [15]:
def visualize_results(df_preprocessed):
    try:
        print("Generating visualizations...")
        if 'Need Medication' in df_preprocessed.columns:
            need_medication = df_preprocessed['Need Medication'].value_counts()
            plt.figure(figsize=(8, 6))
            plt.bar(['No Medication', 'Needs Medication'], need_medication.values, color=['green', 'red'])
            plt.title('People Needing Medical Attention vs. Not')
            plt.ylabel('Count')
            plt.savefig('medical_attention.png')
            plt.close()

        if 'Cost Range' in df_preprocessed.columns:
            cost_range_counts = df_preprocessed['Cost Range'].value_counts()
            plt.figure(figsize=(8, 6))
            plt.pie(cost_range_counts, labels=cost_range_counts.index, autopct='%1.1f%%', colors=['#ff9999', '#66b3ff', '#99ff99', '#ffcc99'])
            plt.title('Percentage of People in Different Price Ranges')
            plt.savefig('price_ranges.png')
            plt.close()

        if 'Sick' in df_preprocessed.columns:
            plt.figure(figsize=(8, 6))
            plt.hist(df_preprocessed['Sick'], bins=2, color='purple', edgecolor='black')
            plt.title('Distribution of Sick vs. Not Sick')
            plt.xlabel('Sick (0 = No, 1 = Yes)')
            plt.ylabel('Count')
            plt.savefig('sick_distribution.png')
            plt.close()

        if 'Age Group' in df_preprocessed.columns and 'Sick' in df_preprocessed.columns:
            plt.figure(figsize=(10, 6))
            sns.countplot(x='Age Group', hue='Sick', data=df_preprocessed)
            plt.title('Sickness Distribution by Age Group')
            plt.xlabel('Age Group')
            plt.ylabel('Count')
            plt.savefig('age_group_sick.png')
            plt.close()

        numerical_cols = df_preprocessed.select_dtypes(include=[np.float64, np.int64]).columns
        if len(numerical_cols) > 1:
            plt.figure(figsize=(12, 8))
            sns.heatmap(df_preprocessed[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
            plt.title('Correlation Heatmap of Numerical Features')
            plt.savefig('correlation_heatmap.png')
            plt.close()

        print("Visualizations saved as images.")
    except Exception as e:
        print(f"Error in visualization: {e}")
        traceback.print_exc()

In [16]:
def combined_analysis(method="upload", file_path=None, url=None, missing_strategy='mean',
                      target_column=None, state_column=None, feature_column=None,
                      bayesian_columns=None, bayesian_edges=None):
    try:
        # Display initial comment block
        print("# Integrated Healthcare Data Analysis Pipeline with Deep Learning and Comprehensive Metrics")
        print("# This script corrects previous errors, uses DiscreteBayesianNetwork, implements custom hyperparameter tuning,")
        print("# and includes explicit Input layers for Keras models. It evaluates accuracy, precision, recall, and F1-score for all models.")
        print("# Expanded to include Enhanced CNN and Enhanced RNN with bidirectional LSTM after neural network with backpropagation.")

        # Load dataset
        df = load_dataset(method=method, file_path=file_path, url=url)
        if df is None:
            return None, None

        # Preprocess dataset
        output_file = "preprocessed_dataset.csv" if method == "upload" else file_path.split('.')[0] + '_preprocessed.csv' if file_path else "preprocessed_dataset.csv"
        df_preprocessed = preprocess_dataset(df, missing_strategy=missing_strategy, output_path=output_file)
        if df_preprocessed is None:
            print(f"Preprocessing failed for {output_file}. Check logs for details.")
            return None, None

        outcome_columns = ['Survive', 'Need Hospitalization', 'Need Medication', 'Need Surgery']

        # Neural Network with Backpropagation
        nn_result = None
        if target_column:
            nn_model, nn_history, nn_y_test, nn_y_pred, nn_metrics = neural_network_with_backpropagation(df_preprocessed, target_column=target_column)
            if nn_model:
                nn_result = (nn_model, nn_history, nn_y_test, nn_y_pred, nn_metrics)
                print("Neural network with backpropagation completed.")

        # Enhanced CNN for Tabular Data
        enhanced_cnn_result = None
        if target_column:
            enhanced_cnn_model, enhanced_cnn_history, enhanced_cnn_y_test, enhanced_cnn_y_pred, enhanced_cnn_metrics = enhanced_cnn_for_tabular(df_preprocessed, target_column=target_column)
            if enhanced_cnn_model:
                enhanced_cnn_result = (enhanced_cnn_model, enhanced_cnn_history, enhanced_cnn_y_test, enhanced_cnn_y_pred, enhanced_cnn_metrics)
                print("Enhanced CNN for tabular data completed.")

        # Enhanced RNN for Sequential Analysis
        enhanced_rnn_result = None
        if target_column:
            enhanced_rnn_model, enhanced_rnn_history, enhanced_rnn_y_test, enhanced_rnn_y_pred, enhanced_rnn_metrics = enhanced_rnn_for_sequential(df_preprocessed, target_column=target_column)
            if enhanced_rnn_model:
                enhanced_rnn_result = (enhanced_rnn_model, enhanced_rnn_history, enhanced_rnn_y_test, enhanced_rnn_y_pred, enhanced_rnn_metrics)
                print("Enhanced RNN for sequential analysis completed.")

        # Custom Hyperparameter Tuning
        grid_result = None
        if target_column:
            grid_model, grid_params, grid_y_test, grid_y_pred, grid_metrics = custom_hyperparameter_tuning(df_preprocessed, target_column=target_column)
            if grid_model:
                grid_result = (grid_model, grid_params, grid_y_test, grid_y_pred, grid_metrics)
                print("Custom hyperparameter tuning completed.")

        # CNN for Tabular Data
        cnn_result = None
        if target_column:
            cnn_model, cnn_history, cnn_y_test, cnn_y_pred, cnn_metrics = cnn_for_tabular(df_preprocessed, target_column=target_column)
            if cnn_model:
                cnn_result = (cnn_model, cnn_history, cnn_y_test, cnn_y_pred, cnn_metrics)
                print("CNN for tabular data completed.")

        # RNN for Sequential Analysis
        rnn_result = None
        if target_column:
            rnn_model, rnn_history, rnn_y_test, rnn_y_pred, rnn_metrics = rnn_for_sequential(df_preprocessed, target_column=target_column)
            if rnn_model:
                rnn_result = (rnn_model, rnn_history, rnn_y_test, rnn_y_pred, rnn_metrics)
                print("RNN for sequential analysis completed.")

        # SVM Classification
        svm_result = None
        if target_column:
            svm_model, svm_y_test, svm_y_pred, svm_metrics = svm_classification(df_preprocessed, target_column=target_column, file_path=output_file)
            if svm_model:
                svm_result = (svm_model, svm_y_test, svm_y_pred, svm_metrics)
                visualize_svm(svm_model, df_preprocessed, target_column)

        # Markov Chain Analysis
        markov_result = None
        if state_column and all(col in df_preprocessed.columns for col in outcome_columns):
            markov_result = markov_chain_analysis(df_preprocessed, state_column=state_column, outcome_columns=outcome_columns)
            print("Markov chain analysis completed.")

        # Monte Carlo Simulations
        monte_carlo_results = {}
        if all(col in df_preprocessed.columns for col in outcome_columns):
            monte_carlo_results['Cost'] = monte_carlo_cost(df_preprocessed)
            monte_carlo_results['Survival'] = monte_carlo_survival(df_preprocessed)
            monte_carlo_results['Hospitalization'] = monte_carlo_hospitalization(df_preprocessed)
            monte_carlo_results['Medication'] = monte_carlo_medication(df_preprocessed)
            monte_carlo_results['Length of Stay'] = monte_carlo_length_of_stay(df_preprocessed)
            monte_carlo_results['Surgery'] = monte_carlo_surgery(df_preprocessed)
            print("Monte Carlo simulations completed for cost, outcomes, length of stay, and surgery.")

        # Bayesian Network Analysis
        bayesian_result = None
        if bayesian_columns and bayesian_edges and all(col in df_preprocessed.columns for col in bayesian_columns):
            bayesian_result = bayesian_network_analysis(df_preprocessed, bayesian_columns, bayesian_edges)
            print("Bayesian network analysis completed.")

        return (nn_result, enhanced_cnn_result, enhanced_rnn_result, grid_result, cnn_result, rnn_result, svm_result, markov_result, monte_carlo_results, bayesian_result), df_preprocessed
    except Exception as e:
        print(f"Error in combined analysis pipeline: {e}")
        traceback.print_exc()
        return None, None

In [ ]:
# Configuration
target_column = "Sick"
state_column = "Cost Range"
feature_column = "Total Costs"
bayesian_columns = ["APR Severity of Illness Code", "APR Risk of Mortality", "Sick"]
bayesian_edges = [("APR Severity of Illness Code", "Sick"), ("APR Risk of Mortality", "Sick")]

# Load via URL (SPARCS)
sample_url = "https://health.data.ny.gov/api/views/22g3-z7e7/rows.csv?accessType=DOWNLOAD"
print(f"\nProcessing SPARCS dataset from URL: {sample_url} at {time.strftime('%H:%M:%S %Z on %Y-%m-%d')}")
results, df_preprocessed = combined_analysis(
    method="url",
    url=sample_url,
    missing_strategy='mean',
    target_column=target_column,
    state_column=state_column,
    feature_column=feature_column,
    bayesian_columns=bayesian_columns,
    bayesian_edges=bayesian_edges
)

# Visualize if successful
if results is not None and df_preprocessed is not None:
    visualize_results(df_preprocessed)


Processing SPARCS dataset from URL: https://health.data.ny.gov/api/views/22g3-z7e7/rows.csv?accessType=DOWNLOAD at 17:52:29 UTC on 2025-10-07
# Integrated Healthcare Data Analysis Pipeline with Deep Learning and Comprehensive Metrics
# This script corrects previous errors, uses DiscreteBayesianNetwork, implements custom hyperparameter tuning,
# and includes explicit Input layers for Keras models. It evaluates accuracy, precision, recall, and F1-score for all models.
# Expanded to include Enhanced CNN and Enhanced RNN with bidirectional LSTM after neural network with backpropagation.
Loaded dataset from URL https://health.data.ny.gov/api/views/22g3-z7e7/rows.csv?accessType=DOWNLOAD with shape (2343569, 34)
Cleaned Total Costs and Total Charges columns
Created binary 'Sick' column based on APR Severity of Illness Code >= 3
Filled 240 NaN values in 'APR Risk of Mortality' with median: 1.0
Created binary 'Survive' column
Created binary 'Need Hospitalization' column
Created binary 'Need Me

Enhanced RNN (Bidirectional LSTM) Evaluation Metrics:
Accuracy: 0.6100
Precision: 0.5731
Recall: 0.6100
F1-Score: 0.5874
Chart for evaluation metrics:
```chartjs
{"type": "bar", "data": {"labels": ["Accuracy", "Precision", "Recall", "F1-Score"], "datasets": [{"label": "Enhanced RNN (Bidirectional LSTM) Metrics", "data": [0.61, 0.5730560578661844, 0.61, 0.5873667587840505], "backgroundColor": ["#FF6384", "#36A2EB", "#FFCE56", "#4BC0C0"], "borderColor": ["#FF6384", "#36A2EB", "#FFCE56", "#4BC0C0"], "borderWidth": 1}]}, "options": {"scales": {"y": {"beginAtZero": true, "title": {"display": true, "text": "Score"}}}, "plugins": {"title": {"display": true, "text": "Enhanced RNN (Bidirectional LSTM) Evaluation Metrics"}}}}
```
Enhanced RNN for Sequential Analysis Metrics:
Test Loss: 2.2992
Enhanced RNN for sequential analysis completed.
